# Session 1 · Part 3 — Spatial neighborhoods and exploratory structure

**Goal:** connect normalized molecular measurements to tissue coordinates. We construct the spatial
kNN graph DGAT-style models consume, visualize its edges, map variable features, and use PCA only as
an exploratory downstream view—not as evidence of cell types by itself.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


## 1. Reload and process the data independently


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.plotting import plot_spatial_feature, plot_spatial_knn_graph
from dgat_tutorial.processing import knn_edge_index, process_modalities

dataset = load_tutorial_data(paths.raw_data)
processed = process_modalities(
    dataset.spots,
    dataset.transcripts.select_dtypes(include=[np.number]),
    dataset.proteins.select_dtypes(include=[np.number]),
)
spots = processed.spots
rna = processed.normalized_transcripts
protein = processed.normalized_proteins
edge_index = knn_edge_index(spots, n_neighbors=6)
print(f"Graph: {len(spots)} nodes, {edge_index.shape[1]} directed edges")


### Figure 3 — The spatial graph passed to a graph encoder

Drawing every edge on all ~4,000 spots makes the local neighborhood structure
unreadable, so this figure splits overview and zoom:

- **Left:** all spots as nodes (no edges), with a red box marking the corner region.
- **Right:** undirected 6-nearest-neighbor edges inside that corner.
- **Highlight:** the red spot is a boundary example; the orange spots are its six
  nearest neighbors. At a tissue corner those neighbors all lie inward, which is
  the local connectivity a graph encoder can attend over.


In [ ]:
fig, _ = plot_spatial_knn_graph(spots, edge_index, n_neighbors=6, zoom_corner="lower_left")
graph_path = paths.figures / "session01_spatial_knn_graph.png"
fig.savefig(graph_path, dpi=160, bbox_inches="tight")
plt.show()


### Figure 4 — Normalized RNA and protein landscapes


In [ ]:
rna_features = rna.var(axis=0).nlargest(3).index
protein_features = protein.var(axis=0).nlargest(3).index
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, feature in zip(axes[0], rna_features):
    plot_spatial_feature(spots, rna[feature], f"Normalized RNA: {feature}", cmap="magma", ax=ax)
for ax, feature in zip(axes[1], protein_features):
    plot_spatial_feature(spots, protein[feature], f"CLR protein: {feature}", cmap="viridis", ax=ax)
figure_path = paths.figures / "session01_normalized_spatial_features.png"
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()


### Figure 5 — Exploratory RNA and protein embeddings

Colors are exploratory KMeans groups in PCA space (**not** curated cell-type labels).

- **Left:** RNA PCA; spots share one of four RNA cluster colors (`0`–`3`).
- **Middle:** protein PCA with its **own** four protein clusters (same palette, independent IDs).
- **Right:** the RNA cluster colors from the left panel mapped back onto tissue coordinates.

Use these panels only to check whether modality structure and spatial organization look coherent before moving to DGAT.


In [ ]:
def pca_clusters(matrix, n_clusters=4):
    embedding = PCA(n_components=2, random_state=7).fit_transform(matrix)
    clusters = KMeans(n_clusters=min(n_clusters, len(matrix)), n_init=20, random_state=7).fit_predict(embedding)
    return embedding, clusters


def cluster_colors(labels, cmap_name="tab10"):
    """Map integer cluster IDs to stable discrete colors."""
    cmap = plt.get_cmap(cmap_name)
    return np.asarray([cmap(int(label) % 10) for label in labels])


rna_pca, rna_cluster = pca_clusters(rna)
protein_pca, protein_cluster = pca_clusters(protein)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
axes[0].scatter(rna_pca[:, 0], rna_pca[:, 1], c=cluster_colors(rna_cluster), s=20)
axes[0].set_title("RNA PCA (exploratory clusters)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

axes[1].scatter(protein_pca[:, 0], protein_pca[:, 1], c=cluster_colors(protein_cluster), s=20)
axes[1].set_title("Protein PCA (exploratory clusters)")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")

axes[2].scatter(spots["x"], spots["y"], c=cluster_colors(rna_cluster), s=20)
axes[2].set_title("RNA clusters mapped to tissue")
axes[2].set_xlabel("x")
axes[2].set_ylabel("y")
axes[2].set_aspect("equal")

embedding_path = paths.figures / "session01_modality_pca_and_spatial_clusters.png"
fig.tight_layout()
fig.savefig(embedding_path, dpi=160, bbox_inches="tight")
plt.show()


## Save the graph and checkpoint


In [ ]:
edge_table = pd.DataFrame({"source": spots.index[edge_index[0]], "target": spots.index[edge_index[1]]})
edge_path = paths.results / "session01_spatial_knn_edges.csv"
edge_table.to_csv(edge_path, index=False)
manifest = write_checkpoint(
    "1.3", [edge_path, graph_path, figure_path, embedding_path],
    summary={"nodes": len(spots), "directed_edges": edge_index.shape[1]}, start=paths.root,
)
print(f"Checkpoint written: {manifest}")


## Check

Trace one spot from its normalized feature vector to its node and six outgoing spatial edges. DGAT's
graph attention layers learn how much neighbor information to aggregate; the graph defines which
neighbors are available.
